In [ ]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix,
)
import json
import os

#NOTE: CONFIG
TEST_CSV  = r""
MODEL_IN  = r""
META_OUT_JSON = r""
META_OUT_CSV  = r""

FEATURES = ["temp_c", "humidity_pct", "lux", "vpd_kpa"]
TARGET = "y"
LABELS = [0, 1, 2]

#NOTE: Load TEST data only
test_df = pd.read_csv(TEST_CSV)
test_df = test_df.dropna(subset=FEATURES + [TARGET])

X_test = test_df[FEATURES].values
y_test = test_df[TARGET].astype(int).values

#NOTE: Load model
model = XGBClassifier()
model.load_model(MODEL_IN)

#NOTE: Evaluate TEST
test_pred = model.predict(X_test)
test_f1 = f1_score(y_test, test_pred, average="macro", labels=LABELS)
test_acc = accuracy_score(y_test, test_pred)
conf_mat = confusion_matrix(y_test, test_pred, labels=LABELS).tolist()
cls_report = classification_report(y_test, test_pred, labels=LABELS, digits=4, output_dict=True)

#NOTE: Save Metadata
metadata = {
    "test_samples": len(y_test),
    "test_class_distribution": test_df[TARGET].value_counts().to_dict(),
    "accuracy": round(test_acc, 4),
    "macro_f1": round(test_f1, 4),
    "confusion_matrix": conf_mat,
    "classification_report": cls_report
}

with open(META_OUT_JSON, "w") as jf:
    json.dump(metadata, jf, indent=4)

pd.DataFrame(cls_report).transpose().to_csv(META_OUT_CSV, index=True)

print("\n=== TEST PERFORMANCE (Never-seen data) ===")
print(f"Accuracy : {test_acc:.4f}")
print(f"Macro-F1 : {test_f1:.4f}")
print("\nConfusion matrix:")
print(conf_mat)
print("\nClassification report:")
print(pd.DataFrame(cls_report).transpose())


=== TEST PERFORMANCE (Never-seen data) ===
Accuracy : 0.9899
Macro-F1 : 0.6279

Confusion matrix:
[[94, 0, 0], [1, 4, 0], [0, 0, 0]]

Classification report:
              precision    recall  f1-score    support
0              0.989474  1.000000  0.994709  94.000000
1              1.000000  0.800000  0.888889   5.000000
2              0.000000  0.000000  0.000000   0.000000
accuracy       0.989899  0.989899  0.989899   0.989899
macro avg      0.663158  0.600000  0.627866  99.000000
weighted avg   0.990005  0.989899  0.989365  99.000000


c:\CodeKubbb\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\CodeKubbb\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\CodeKubbb\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\CodeKubbb\.venv\Lib\site-packages\sklearn\me